# Crypto oscillator finder — NONKYC v11 (`recycle_v11` engine)

**v11.1** adds: a dust filter (`min_vol_usd`), quote-unit conversion for crypto-quoted
pairs (fund / min-order / volume-cap all in the pair's quote currency; exports carry both
`max_fund_value_quote` in quote units and `max_fund_value_usd`), finalize reusing the same
order-book snapshot the engine already sampled (no spurious spread gates), the
`rc_gate_two_sided_mode` holdout waiver, and a book-verification cell (Cell 10).

**What changed vs v10** (full details in `LadderLab_v11_notes_claude_code.md`):

1. **True holdout** — the deploy ladder is fitted *without* the last 15 days; the fitted,
   train-anchored ladder is scored once on that unseen tail. `holdout_edge_pct` is the only
   fully out-of-sample number for the geometry you deploy, and it gates deployment.
2. **Clean-block gating** — the 180d frozen report tags blocks that overlap the fit window
   (`in_fit`); the band-aware v10 gates must now pass on the *clean* blocks too.
3. **Fill realism** — fills require price to trade *through* the rung
   (`rc_fill_penetration_pct` + 1 tick), and per-bar filled notional is capped at
   `rc_volume_cap_frac` × the bar's quote volume, with partial fills (bars now carry volume).
4. **Slip symmetry** — the measured live order-book half-spread joins Roll + floor in the
   *search and WF*, not just as a finalize gate.
5. **Grid-harvest screen** — a model-free zig-zag swing count (net of round-trip cost) ranks
   the review set; it cannot overfit because nothing is fitted.
6. **Deep history** — MEXC-first Nx6 (with volume) history (NonKYC-native 5m first for report bars).

Nothing was removed from the v10 reports — v11 only adds columns/files. The
`*_copy_paste_ladders.md` is rendered by v10's renderer, byte-identical format.


In [1]:
# ── Cell 1: bootstrap ────────────────────────────────────────────────
# numba makes the engine ~50-100x faster; strongly recommended.
import sys, subprocess
for _mod, _pkg in (('numba', 'numba'), ('tabulate', 'tabulate')):
    try:
        __import__(_mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=False)

import importlib, os, time, json
from pathlib import Path
import numpy as np
import pandas as pd

import ladder_lab as ll
import ladder_lab_recycle as lr10          # v10 stays installed; v11 reuses it
import ladder_lab_recycle_v11 as lr
importlib.reload(ll); importlib.reload(lr10); importlib.reload(lr)

EXCHANGE = 'nonkyc'

print(f'ladder_lab {ll.__version__} | v10 {lr10.__version__} | v11 {lr.__version__} | numba={ll.HAVE_NUMBA}')
assert ll.parity_check(), 'base kernel parity FAILED'
assert lr10.recycle_parity_check(), 'v10 kernel parity FAILED'
assert lr.recycle_v11_parity_check(verbose=True), 'v11 kernel parity / v10-equivalence FAILED'
print('kernel parity OK (v11 reproduces v10 exactly with realism features off)')

CFG = lr.recycle_default_config(EXCHANGE)
# ---- knobs you will most likely touch ----
CFG['always_review_markets'] = ('XMR/USDT', 'XMR/USD', 'SAL/USDT')
CFG['focus_markets']         = ('ZANO/USDT', 'DASH/USDT', 'SUN/USDT')
CFG['rc_top_n_from_screen']  = 40      # markets from the screener that get the full engine
CFG['rc_n_candidates']       = 240     # per fit; drop to ~80 if running without numba
CFG['rc_target_trades_per_15d'] = 40   # recalibrate with the diagnostic-JSONL cell at the bottom
CFG['rc_report_interval']    = '5m'    # frozen reports / YAML eval / holdout on 5m bars
# ---- v11 realism knobs (defaults are the recommended settings) ----
CFG['rc_holdout_days']         = 15    # fit never sees the last N days (0 disables; don't)
CFG['rc_fill_penetration_pct'] = 0.0005  # price must trade 5bps+1tick THROUGH a rung to fill
CFG['rc_volume_cap_frac']      = 0.25  # max share of a bar's quote volume we may fill
CFG['rc_book_spread_in_slip']  = True  # live half-spread joins Roll+floor in search/WF too
# ---- v11.1 knobs ----
CFG['min_vol_usd']             = 10_000  # DUST FILTER: drop markets under this 24h USD volume
                                         # (the first run spent ~90% of engine time on books
                                         #  that could not absorb even the $200 fund floor)
CFG['rc_gate_two_sided_mode']  = 'either'  # 'blocks'=v10 | 'holdout' | 'either' (waive the
                                         # block two-sided gate when the HOLDOUT was two-sided;
                                         # re-anchored ladders are structurally one-sided vs a
                                         # trending past -- the holdout has no such bias)
# CFG['rc_v11_fill_model'] = False     # A/B: byte-identical v10 fills (leave True to deploy)

# Where your live controller YAMLs live (searched recursively):
YAML_ROOTS = ['.', 'controllers', str(Path.home())]
YAML_ROOTS = ['./controllers/nonkyc']

# artifacts/{exchange}/{timestamp}/files
RUN_TS = time.strftime('%Y%m%d-%H%M%S')
ART = Path('artifacts') / EXCHANGE / RUN_TS / 'files'
ART.mkdir(parents=True, exist_ok=True)
PREFIX = str(ART / f'NONKYC_recycle_v11')
print('artifacts →', ART)


ladder_lab 1.0.0 | v10 10.2.0-recycle-rollspread-bandgates | v11 11.3.1-resilient-universe | numba=True
  series 0: v10-equiv=OK trades=20
  series 1: v10-equiv=OK trades=27
  series 2: v10-equiv=OK trades=18
  series 3: v10-equiv=OK trades=17
kernel parity OK (v11 reproduces v10 exactly with realism features off)
artifacts → artifacts/nonkyc/20260713-040416/files


In [2]:
# ── Cell 2: universe ─────────────────────────────────────────────────
# v11.1.3: build the universe WITHOUT the volume floor so live-YAML and
# always-review markets can never be starved of history (a live ZANO ladder
# was silently dropped when its 24h volume dipped under the floor). The dust
# filter is applied in Cell 4 to the SCREENED candidates only -- engine time
# is still protected, your own markets always get evaluated.
# v11.3.1: patient retries (15/30/45/60s -- outlasts a rate-limit window)
# and a loud fallback to the last good universe cached on disk (<=48h old),
# so a flaky /market/getlist costs freshness, not the whole run.
uni = lr.build_universe(EXCHANGE, CFG)
VOL_USD = dict(zip(uni['df'].pairkey, uni['df'].vol_usd))
cache = ll.CandleCache(CFG['cache_dir'], CFG['cache_ttl_hours'])


338 nonkyc markets across 13 quote(s) >= $0 24h vol (USD-equiv).
  by quote: USDT:231, USDC:35, BTC:32, XMR:11, ETH:10, DOGE:7, ZSD:3, BNB:3, ERG:2, FUSD:1, LTC:1, SOL:1, EGAZ:1


In [3]:
# ── Cell 3: daily history + screener (breadth pass, unchanged from base) ──
hist_d = ll.prefetch_history(uni, CFG, cache)
screen_df = ll.screen(uni, hist_d, CFG)
screen_df = lr.drop_stable_stable(screen_df, CFG)
screen_df.head(25)


  ...MEXC pass 50/338
  ...MEXC pass 100/338
  ...MEXC pass 150/338
  ...MEXC pass 200/338
  ...MEXC pass 250/338
  ...MEXC pass 300/338
  native fallback for 212 markets (small pool) ...
history: 326/338 markets in 124s (MEXC 126, native 200, missing 12)
  dropped 4 stable/stable pairs from consideration


,days,price,low,high,ann_vol,er,cross_per_yr,in_band,net_return,cur_vs_high,...,coin,quote,src,vol_usd,min_qty,stale,tier,range_x,qualifies,composite
0,184,7.000000e-06,5.814500e-06,1.183550e-05,3.494227,0.015285,91.250000,0.597826,-0.263933,0.549882,...,SATOX,USDT,NonKYC,5.38,0.0,False,full,2.04,True,0.979
1,217,9.129100e-04,6.799600e-04,9.129100e-04,3.083984,0.060747,77.373272,0.599078,0.292634,0.906501,...,SLDX,USDT,NonKYC,8.05,0.0,False,full,1.34,True,0.977
2,217,1.686000e-07,1.625600e-07,2.642000e-07,3.231886,0.040795,67.281106,0.603687,-0.361847,0.140512,...,PDOGE,USDT,NonKYC,47.59,0.0,False,full,1.63,True,0.969
3,220,1.675000e-01,1.600950e-01,3.901750e-01,2.973905,0.055147,69.681818,0.604545,-0.630813,0.260863,...,ZCL,USDT,NonKYC,152.26,0.0,False,full,2.44,True,0.957
4,220,1.288000e-03,1.085090e-03,1.553760e-03,1.594886,0.022852,81.295455,0.600000,-0.222739,0.697989,...,DIVI,USDT,NonKYC,31713.52,0.0,False,full,1.43,True,0.945
5,220,9.771640e-04,1.051907e-03,2.379889e-03,4.160283,0.051000,49.772727,0.600000,-0.513351,0.299164,...,PEP,DOGE,NonKYC,3024.61,0.0,False,full,2.26,True,0.939
6,218,6.156900e-02,6.470555e-02,1.040195e-01,12.508760,0.126623,41.857798,0.605505,-0.403534,0.591168,...,CHESS,USDC,NonKYC,679.94,0.0,False,full,1.61,True,0.935
7,217,7.810000e-09,6.316000e-09,1.055800e-08,3.282188,0.002490,45.414747,0.603687,0.044118,0.282051,...,DOGS,USDT,NonKYC,4.14,0.0,False,full,1.67,True,0.922
8,217,2.908600e-10,2.837460e-10,3.633000e-10,6.511288,0.000090,33.640553,0.594470,-0.000550,0.739612,...,PDOGE,BNB,NonKYC,4.38,0.0,False,full,1.28,True,0.919
9,219,5.021900e-06,5.006040e-06,1.153501e-05,2.564428,0.041049,55.000000,0.598174,-0.517097,0.379507,...,XLA,USDT,NonKYC,28.56,0.0,False,full,2.30,True,0.913


In [4]:
# ── Cell 4: pick the markets that get the full engine ────────────────
# v11: the model-free grid-harvest rank (zig-zag swings net of round-trip cost)
# is blended with the base composite. harvest_* columns are on DAILY bars here
# (breadth); the engine recomputes them on intraday bars for the finalists.
yamls = lr.discover_controller_yamls(YAML_ROOTS, exchange=EXCHANGE)
yaml_markets = [lr._pair_key(p['trading_pair']) for p in yamls]
print(f'controller YAMLs found: {len(yamls)} → {yaml_markets}')

ranked = lr.rank_review_markets(screen_df, hist_d, CFG)
display(ranked.head(25))

col = 'base' if 'base' in ranked.columns else 'pairkey'
# dust filter applies HERE (screened candidates only); YAML/always/focus bypass
eligible = [pk for pk in ranked[col] if VOL_USD.get(pk, 0) >= CFG['min_vol_usd']]
dropped = len(ranked) - len(eligible)
if dropped:
    print(f'dust filter: {dropped} screened markets under ${CFG["min_vol_usd"]:,.0f} 24h volume')
top_screen = eligible[:CFG['rc_top_n_from_screen']]
wanted = lr.merge_unique_markets(top_screen, CFG['always_review_markets'],
                                 CFG['focus_markets'], yaml_markets)
review, missing = lr.resolve_present(uni, wanted)
if missing:
    print('not on this exchange / below volume floor:', missing)
print(f'{len(review)} markets selected for the v11 engine')


controller YAMLs found: 4 → ['DASH/USDT', 'SUN/USDT', 'XMR/USDT', 'ZANO/USDT']


,base,days,price,low,high,ann_vol,er,cross_per_yr,in_band,net_return,...,range_x,qualifies,composite,harvest_rt_cost_pct,harvest_1.5x_pct_mo,harvest_2x_pct_mo,harvest_3x_pct_mo,harvest_best_pct_mo,harvest_best_gap_pct,review_rank
0,SATOX/USDT,184,7.000000e-06,5.814500e-06,1.183550e-05,3.494227,0.015285,91.250000,0.597826,-0.263933,...,2.04,True,0.979,12.400,15.960,21.280,13.680,21.280,24.800,0.970
1,BCA/USDT,220,3.612900e-02,3.060485e-02,6.463095e-02,2.350918,0.018994,49.772727,0.613636,-0.298316,...,2.11,True,0.904,8.670,12.487,21.786,22.318,22.318,26.009,0.960
2,ZCL/USDT,220,1.675000e-01,1.600950e-01,3.901750e-01,2.973905,0.055147,69.681818,0.604545,-0.630813,...,2.44,True,0.957,12.400,13.680,17.480,19.760,19.760,37.200,0.954
3,SHA/BTC,220,2.548970e-07,1.766926e-07,6.350266e-07,3.230323,0.017606,51.431818,0.600000,-0.382572,...,3.59,True,0.865,11.599,13.863,22.039,24.171,24.171,34.798,0.948
4,CCX/BTC,220,9.800000e-08,8.690000e-08,1.901000e-07,1.819000,0.047778,53.090909,0.604545,-0.467391,...,2.19,True,0.865,5.787,7.981,13.834,20.573,20.573,17.362,0.928
5,DOSU/DOGE,220,8.280000e-09,6.174000e-09,1.489500e-08,2.606933,0.037758,41.477273,0.600000,-0.442049,...,2.41,True,0.875,11.428,9.806,14.009,19.613,19.613,34.285,0.927
6,BCH2/USDT,113,1.586542e-02,1.588814e-02,9.898972e-02,3.234294,0.187775,51.681416,0.592920,-0.880189,...,6.23,False,0.787,10.876,16.975,30.998,32.474,32.474,32.629,0.925
7,CCX/USDT,220,6.232000e-03,6.294350e-03,1.575680e-02,1.773079,0.078510,69.681818,0.600000,-0.633649,...,2.50,True,0.886,7.329,9.208,14.823,18.865,18.865,21.986,0.925
8,IXI/USDT,219,1.284000e-05,8.089000e-06,2.009800e-05,2.505741,0.018495,51.666667,0.607306,-0.256944,...,2.48,True,0.910,11.420,9.799,16.799,16.799,16.799,22.840,0.925
9,FLOP/USDT,220,3.518000e-07,2.087950e-07,9.836150e-07,3.217507,0.019196,33.181818,0.600000,-0.360945,...,4.71,True,0.784,12.400,14.440,22.800,31.920,31.920,37.200,0.922


dust filter: 264 screened markets under $10,000 24h volume
not on this exchange / below volume floor: ['XMR/USD']
42 markets selected for the v11 engine


In [5]:
# ── Cell 5: history — hourly Nx6 for search/WF/holdout, 5m Nx6 for reports ─
# v11 bars carry QUOTE VOLUME (column 6) so the sim can cap fills against what
# actually traded. Kraken pairs ride the MEXC proxy (USD→USDT alias, guarded by
# the last-price check); Kraken-native OHLC is a 30d fallback (720-candle cap).
hist_h = lr.prefetch_bars6(uni, CFG, cache, pairs=review, interval='60m',
                           min_days_key='rc_min_hourly_days')
hist_h = lr.with_daily_fallback(hist_h, hist_d, CFG, review)   # daily fallback is flagged
print(pd.Series({pk: v['granularity'] for pk, v in hist_h.items()}).value_counts().to_string())
vol_missing = [pk for pk, v in hist_h.items() if not v.get('vol_known', False)]
if vol_missing:
    print('volume unknown (volume cap disabled) for:', vol_missing)

if CFG['rc_report_interval'] not in ('60m', '1h'):
    hist_report = lr.prefetch_bars6(uni, CFG, cache, pairs=review,
                                    interval=CFG['rc_report_interval'],
                                    min_days_key='rc_min_intraday_days')
    print(pd.Series({pk: v['granularity'] for pk, v in hist_report.items()}).value_counts().to_string())
else:
    hist_report = hist_h
missing_5m = [pk for pk in review if pk in hist_h and pk not in hist_report]
if missing_5m:
    print('report falls back to search-granularity bars for:', missing_5m)


  ...bars6[60m] 10/42
  ...bars6[60m] 20/42
  ...bars6[60m] 30/42
  ...bars6[60m] 40/42
bars6(60m): 42/42 markets in 1s
1h    42
  ...bars6[5m] 10/42
  ...bars6[5m] 20/42
  ...bars6[5m] 30/42
  ...bars6[5m] 40/42
bars6(5m): 42/42 markets in 1s
5m    42


In [6]:
# ── Cell 6: YOUR live controller YAMLs, first-class, on the same engine ──
# Genuinely out-of-sample for any ladder you wrote before this window ends.
# v11 engine: penetration + volume caps + book-spread slip apply here too, so
# these numbers are the closest thing to a live replay in the whole pipeline.
yaml_hist = {pk: hist_report.get(pk, hist_h.get(pk)) for pk in set(list(hist_h) + list(hist_report))}
yaml_hist = {pk: v for pk, v in yaml_hist.items() if v is not None}
# Mark which YAMLs are LIVE (ground truth) vs untraded DRAFTS (hypotheses).
# Leave empty to treat them all as live; otherwise list deployed controller_ids.
DEPLOYED_CONTROLLERS = set()      # e.g. {'range_inventory_ladder_xmr_V1'}
yaml_df, yaml_reports = lr.evaluate_controller_yamls(yamls, yaml_hist, uni, CFG)
if not yaml_df.empty and 'controller' in yaml_df.columns and DEPLOYED_CONTROLLERS:
    yaml_df.insert(1, 'status', yaml_df.controller.map(
        lambda c: 'LIVE' if c in DEPLOYED_CONTROLLERS else 'DRAFT'))
    print('LIVE = deployed (ground truth) | DRAFT = never traded (a backtest of a proposal)')
if not yaml_df.empty:
    display(yaml_df)
    # ---- v11.3: LIVE-STRATEGY HEALTH -- recent form, judged on two axes ----
    # A 180-day headline can hide a strategy that died a month ago. This is
    # the table that makes "no longer profitable" impossible to miss.
    health = lr.live_strategy_health(yaml_df, yaml_reports, CFG)
    print()
    lr.print_health_banners(health)
    display(health)
    health.to_csv(f'{PREFIX}_live_health.csv', index=False)
    for cid, rep in yaml_reports.items():
        print(f"\n=== {cid} — passed={rep['passed']}"
              f"  {('; '.join(rep['failed_gates']) if rep['failed_gates'] else '')} ===")
        display(rep['blocks'])
else:
    print('no controller YAMLs matched this exchange — set YAML_ROOTS in Cell 1')


,controller,pair,src,days,blocks,pnl_pct,hold_pct,edge_pct,edge_pos_rate,abs_pos_rate,worst_block_edge,trades,trades_per_month,med_trades_per_block,maxdd,endinv,stress_edge_pct,vol_capped_fills,passed,failed_gates
0,range_inventory_ladder_dash_V1,DASH/USDT,NonKYC,180.0,12,11.262,-12.139,23.402,0.909,0.727,-5.400,219,37.0,17.0,29.814,3.0,20.904,65,True,
1,range_inventory_ladder_sun_V1,SUN/USDT,NonKYC,170.4,11,5.561,-10.066,15.627,0.700,0.636,-6.435,129,23.0,13.0,18.053,10.5,12.250,33,False,two_sided_rate 0.36 < 0.5 (active blocks); cle...
2,range_inventory_ladder_xmr_V1,XMR/USDT,NonKYC,180.0,12,-9.067,-43.524,34.457,0.727,0.778,-7.395,743,125.5,81.0,54.944,85.1,25.793,1,True,
3,range_inventory_ladder_zano_V1,ZANO/USDT,NonKYC,180.0,12,40.167,-9.256,49.423,0.778,0.455,-4.759,153,25.8,12.0,47.296,61.7,35.218,26,False,abs_pos_rate 0.45 < 0.5 (active blocks); clean...


,controller,pair,status,recent_blocks,recent_days,recent_pnl_pct,recent_hold_pct,recent_edge_pct,recent_trades,recent_two_sided_blocks,last_block_pnl_pct,full_pnl_pct,full_edge_pct
0,range_inventory_ladder_dash_V1,DASH/USDT,HEALTHY,3,45.0,6.25,-2.95,9.20,65,3,1.410,11.262,23.402
1,range_inventory_ladder_sun_V1,SUN/USDT,HEALTHY,3,45.0,7.47,-8.59,16.05,32,1,7.332,5.561,15.627
2,range_inventory_ladder_zano_V1,ZANO/USDT,HEALTHY,3,45.0,16.44,-1.57,18.00,38,3,6.872,40.167,49.423
3,range_inventory_ladder_xmr_V1,XMR/USDT,HEALTHY,3,45.0,18.92,-6.67,25.59,240,3,6.887,-9.067,34.457



=== range_inventory_ladder_dash_V1 — passed=True   ===


,block,start,end,days,pnl_pct,hold_pct,edge_pct,buy_fills,sell_fills,trades,two_sided,in_band_pct,in_fit,partial
0,1,2026-01-13,2026-01-28,15.0,1.874,1.874,0.000,0,0,0,False,0.000,False,False
1,2,2026-01-28,2026-02-12,15.0,-13.983,-14.625,0.642,3,7,10,True,0.810,False,False
2,3,2026-02-12,2026-02-27,15.0,0.625,0.072,0.552,8,15,23,True,1.000,False,False
3,4,2026-02-27,2026-03-14,15.0,3.633,-0.184,3.817,21,23,44,True,1.000,False,False
4,5,2026-03-14,2026-03-29,15.0,0.558,-0.927,1.485,18,7,25,True,1.000,False,False
5,6,2026-03-29,2026-04-13,15.0,17.390,6.159,11.231,25,12,37,True,0.963,False,False
6,7,2026-04-13,2026-04-28,15.0,-3.376,-3.958,0.581,0,14,14,False,1.000,False,False
7,8,2026-04-28,2026-05-13,15.0,0.491,5.891,-5.400,0,1,1,False,0.527,False,False
8,9,2026-05-13,2026-05-28,15.0,-0.250,-2.809,2.560,0,0,0,False,0.871,False,False
9,10,2026-05-28,2026-06-12,15.0,4.539,-2.568,7.107,6,3,9,True,1.000,False,False



=== range_inventory_ladder_sun_V1 — passed=False  two_sided_rate 0.36 < 0.5 (active blocks); clean: two_sided_rate 0.36 < 0.5 (active blocks) ===


,block,start,end,days,pnl_pct,hold_pct,edge_pct,buy_fills,sell_fills,trades,two_sided,in_band_pct,in_fit,partial
0,1,2026-01-23,2026-02-07,15.0,-12.874,-19.067,6.193,5,11,16,True,1.000,False,False
1,2,2026-02-07,2026-02-22,15.0,5.080,3.852,1.228,6,8,14,True,1.000,False,False
2,3,2026-02-22,2026-03-09,15.0,-3.213,-3.714,0.500,13,0,13,False,0.938,False,False
3,4,2026-03-09,2026-03-24,15.0,12.167,13.581,-1.414,4,10,14,True,1.000,False,False
4,5,2026-03-24,2026-04-08,15.0,-2.949,-6.353,3.404,0,13,13,False,1.000,False,False
5,6,2026-04-08,2026-04-23,15.0,0.258,6.693,-6.435,0,3,3,False,1.000,False,False
6,7,2026-04-23,2026-05-08,15.0,0.154,5.614,-5.460,0,0,0,False,1.000,False,False
7,8,2026-05-08,2026-05-23,15.0,0.001,-0.034,0.035,0,1,1,False,1.000,False,False
8,9,2026-05-23,2026-06-07,15.0,-0.243,-10.439,10.196,0,0,0,False,1.000,False,False
9,10,2026-06-07,2026-06-22,15.0,0.369,-1.788,2.157,3,0,3,False,1.000,False,False



=== range_inventory_ladder_xmr_V1 — passed=True   ===


,block,start,end,days,pnl_pct,hold_pct,edge_pct,buy_fills,sell_fills,trades,two_sided,in_band_pct,in_fit,partial
0,1,2026-01-13,2026-01-28,15.0,-24.221,-24.221,0.000,0,0,0,False,0.000,False,False
1,2,2026-01-28,2026-02-12,15.0,-14.184,-24.696,10.512,71,88,159,True,0.550,False,False
2,3,2026-02-12,2026-02-27,15.0,8.823,1.884,6.939,44,86,130,True,1.000,False,False
3,4,2026-02-27,2026-03-14,15.0,0.680,3.716,-3.036,3,25,28,True,1.000,False,False
4,5,2026-03-14,2026-03-29,15.0,1.534,-6.303,7.837,20,24,44,True,1.000,False,False
5,6,2026-03-29,2026-04-13,15.0,5.840,4.263,1.577,55,82,137,True,1.000,False,False
6,7,2026-04-13,2026-04-28,15.0,0.173,7.568,-7.395,0,5,5,False,0.834,False,False
7,8,2026-04-28,2026-05-13,15.0,0.110,5.262,-5.151,0,0,0,False,0.164,False,False
8,9,2026-05-13,2026-05-28,15.0,-0.200,-9.083,8.883,0,0,0,False,0.109,False,False
9,10,2026-05-28,2026-06-12,15.0,12.641,-0.469,13.110,40,45,85,True,0.835,False,False



=== range_inventory_ladder_zano_V1 — passed=False  abs_pos_rate 0.45 < 0.5 (active blocks); clean: abs_pos_rate 0.45 < 0.5 (active blocks) ===


,block,start,end,days,pnl_pct,hold_pct,edge_pct,buy_fills,sell_fills,trades,two_sided,in_band_pct,in_fit,partial
0,1,2026-01-13,2026-01-28,15.0,-3.338,-15.795,12.457,26,15,41,True,0.976,False,False
1,2,2026-01-28,2026-02-12,15.0,-10.796,-10.995,0.199,5,0,5,False,0.943,False,False
2,3,2026-02-12,2026-02-27,15.0,-5.256,-5.284,0.028,0,0,0,False,0.837,False,False
3,4,2026-02-27,2026-03-14,15.0,-24.305,-24.441,0.137,0,0,0,False,0.000,False,False
4,5,2026-03-14,2026-03-29,15.0,66.158,66.650,-0.492,0,0,0,False,0.679,False,False
5,6,2026-03-29,2026-04-13,15.0,-2.138,-6.241,4.103,18,4,22,True,1.000,False,False
6,7,2026-04-13,2026-04-28,15.0,6.256,3.452,2.803,13,4,17,True,1.000,False,False
7,8,2026-04-28,2026-05-13,15.0,13.511,18.270,-4.759,12,10,22,True,0.814,False,False
8,9,2026-05-13,2026-05-28,15.0,-0.627,-9.990,9.363,0,8,8,False,0.690,False,False
9,10,2026-05-28,2026-06-12,15.0,-1.326,-10.677,9.352,8,4,12,True,1.000,False,False


In [7]:
# ── Cell 7: full engine per market (WF + HOLDOUT fit + frozen block report) ──
evals = {}
t0 = time.time()
for i, pk in enumerate(review, 1):
    if pk not in hist_h:
        print(f'[{i}/{len(review)}] {pk}: no usable history'); continue
    try:
        ev = lr.evaluate_market(pk, hist_h, uni, CFG, hist_report=hist_report)
    except Exception as e:
        print(f'[{i}/{len(review)}] {pk}: ERROR {e}'); continue
    evals[pk] = ev
    s, wf, ho = ev['report']['summary'], ev['wf'], ev.get('holdout') or {}
    print(f"[{i}/{len(review)}] {pk:14s} rep={ev['granularity']}/fit={ev['search_granularity']} "
          f"edge={s['edge_pct']:+7.2f}% epr={s.get('edge_pos_rate')} "
          f"clean_epr={s.get('clean_edge_pos_rate')} "
          f"HOLDOUT={ho.get('edge_pct', float('nan')):+.2f}% "
          f"blocks_pass={ev['report']['passed']} wf_pass={wf.get('wf_pass')}")
print(f'engine done in {time.time()-t0:.0f}s')


[1/42] BCH2/USDT      rep=5m/fit=1h edge= +24.90% epr=0.8 clean_epr=None HOLDOUT=+8.85% blocks_pass=False wf_pass=False
[2/42] DIVI/USDT      rep=5m/fit=1h edge= +68.41% epr=0.667 clean_epr=0.571 HOLDOUT=-1.40% blocks_pass=False wf_pass=False
[3/42] SAL/USDT       rep=5m/fit=1h edge=  +3.81% epr=0.5 clean_epr=1.0 HOLDOUT=-13.77% blocks_pass=False wf_pass=False
[4/42] BELLS/BTC      rep=5m/fit=1h edge= +47.68% epr=0.8 clean_epr=0.714 HOLDOUT=-1.72% blocks_pass=True wf_pass=True
[5/42] ARRR/USDT      rep=5m/fit=1h edge= +30.70% epr=0.7 clean_epr=0.6 HOLDOUT=-0.93% blocks_pass=False wf_pass=False
[6/42] BELLS/USDT     rep=5m/fit=1h edge= +43.58% epr=0.7 clean_epr=0.8 HOLDOUT=+17.43% blocks_pass=True wf_pass=True
[7/42] PEP/USDC       rep=5m/fit=1h edge=  +5.11% epr=1.0 clean_epr=1.0 HOLDOUT=-4.00% blocks_pass=False wf_pass=False
[8/42] PYTH/USDT      rep=5m/fit=1h edge= +60.22% epr=0.75 clean_epr=0.857 HOLDOUT=-13.05% blocks_pass=False wf_pass=False
[9/42] XMR/USDC       rep=5m/fit=1h edg

In [8]:
# ── Cell 8: rolling walk-forward summary (secondary, leakage-safe) ────
wf_rows = pd.DataFrame([dict(market=pk, **{k: v for k, v in ev['wf'].items() if k != 'folds'})
                        for pk, ev in evals.items() if ev])
if not wf_rows.empty:
    wf_rows = wf_rows.sort_values(['wf_pass', 'edge_pos_rate', 'median_edge_pct'],
                                  ascending=[False, False, False]).reset_index(drop=True)
display(wf_rows)


,market,n_folds,edge_pos_rate,two_sided_rate,median_edge_pct,median_test_pct,median_stress_edge,worst_edge_pct,median_trades,total_trades,wf_pass
0,BELLS/BTC,8,0.875,0.875,1.779,1.747,0.930,-0.039,30.0,213,True
1,NKYC/USDT,8,0.875,0.625,1.381,0.756,0.691,0.000,24.0,252,True
2,W/USDT,6,0.833,0.667,1.793,-1.075,1.728,-5.468,31.0,207,True
3,BELLS/USDT,8,0.750,0.875,5.105,3.966,3.651,-4.143,31.0,294,True
4,RENDER/USDT,8,0.625,0.500,2.361,2.983,1.728,-7.696,20.0,184,True
5,LINK/USDT,8,0.625,0.625,1.486,1.648,-0.669,-3.840,20.0,174,True
6,XMR/USDC,8,0.625,0.625,1.350,1.154,0.535,-4.258,23.0,190,True
7,DASH/USDT,8,0.625,0.750,1.181,4.222,-1.543,-5.603,21.0,212,True
8,XMR/USDT,8,0.625,0.500,0.690,0.938,0.625,-4.759,25.5,206,True
9,BDX/USDT,8,0.625,0.500,0.407,0.496,0.172,-2.222,36.5,303,True


In [9]:
# ── Cell 9: finalize (v10 gates + holdout/clean gates + sizing), write artifacts ─
final_df, configs = lr.finalize_v11(evals, uni, CFG, yaml_reports)
display(final_df)

print('\n— HOLDOUT (the deciding out-of-sample evidence) —')
ho_view = final_df[['base', 'validation', 'holdout_edge_pct', 'holdout_trades',
                    'holdout_two_sided', 'holdout_active', 'edge_pct',
                    'clean_edge_pos_rate', 'harvest_best_pct_mo',
                    'max_fund', 'max_fund_quote', 'depth_2pct', 'gates']]
display(ho_view)

block_tables = {pk: ev['report']['blocks'] for pk, ev in evals.items() if ev}
written = lr.save_v11_outputs(PREFIX, final_df, configs, wf_rows, yaml_df, block_tables,
                              metadata=dict(exchange=EXCHANGE, run=RUN_TS,
                                            cfg={k: str(v) for k, v in CFG.items() if str(k).startswith('rc_')},
                                            yaml_paths=[p['path'] for p in yamls]))
# ---- v11.3: FULLY DEPLOYABLE YAMLs ----------------------------------------
# One ready-to-run controller config per market that is CONFIRMED, a strong
# CANDIDATE (GATED on the WF process check only, with a two-sided holdout and
# clean blocks), or ALREADY LIVE (a REFRESH re-fit at today's anchor -- never
# drop one over a running config without comparing rungs; bump the id and
# start a clean state file if you adopt it).
deployed_pairs = [lr._pair_key(p['trading_pair']) for p in yamls
                  if not DEPLOYED_CONTROLLERS
                  or p.get('controller_id', p.get('id')) in DEPLOYED_CONTROLLERS]
yaml_paths = lr.save_deploy_yamls(ART / 'deploy_yamls', configs, final_df, CFG,
                                  deployed_pairs=deployed_pairs, run_id=RUN_TS[:8])
print(f'\n{len(yaml_paths)-1} deployable YAMLs -> {ART}/deploy_yamls (see INDEX.md)')
print('Read:', f'{PREFIX}_copy_paste_ladders.md')


,base,trading_pair,src,granularity,validation,rungs,family,spacing,weights,pnl_pct,...,quote_usd_rate,max_fund_quote,max_fund,spread_pct,depth_2pct,depth_used,depth_basis,depth_ladder_band,book_total_usd,gates
0,DASH/USDT,DASH-USDT,NonKYC,5m,CONFIRMED,7+9,quantile,back_loaded,slight_deep,37.414,...,1.0000,1000.000000,1000,0.8417,2.0,30066.0,ladder_band,30066.0,33535.0,
1,BELLS/USDT,BELLS-USDT,NonKYC,5m,CONFIRMED,4+9,quantile,geometric,slight_deep,27.777,...,1.0000,700.000000,700,1.0127,311.0,1394.0,ladder_band,1394.0,12723.0,
2,BELLS/BTC,BELLS-BTC,NonKYC,5m,GATED,4+10,quantile,front_loaded,mild_near,52.649,...,63449.9900,0.003152,200,1.0695,3.0,55.0,min_band_5pct,12.0,7076.0,thin book ($55 in the min_band_5pct)
3,FET/USDT,FET-USDT,NonKYC,5m,GATED,7+4,volatility,geometric,near,13.485,...,1.0000,300.000000,300,0.0632,25090.0,47826.0,ladder_band,47826.0,47852.0,rolling WF (process check) failed
4,BC2/USDT,BC2-USDT,NonKYC,5m,SUSPECT,10+5,pct,back_loaded,near,-13.616,...,1.0000,1000.000000,1000,1.1885,676.0,1914.0,ladder_band,1914.0,8654.0,only 2 ACTIVE blocks (< 4) -- band rarely visi...
5,EPIC/USDT,EPIC-USDT,NonKYC,5m,SUSPECT,5+7,volatility,geometric,mild_near,6.904,...,1.0000,600.000000,600,0.8728,23.0,5807.0,ladder_band,5807.0,63420.0,clean: only 1 ACTIVE blocks (< 4) -- band rare...
6,PEP/USDC,PEP-USDC,NonKYC,5m,SUSPECT,7+9,volatility,back_loaded,slight_deep,-31.340,...,1.0029,199.422000,200,1.0160,7.0,161.0,ladder_band,161.0,141130.0,only 2 ACTIVE blocks (< 4) -- band rarely visi...
7,PEP/USDT,PEP-USDT,NonKYC,5m,SUSPECT,10+8,volatility,back_loaded,slight_deep,-32.350,...,1.0000,800.000000,800,0.1572,138.0,3378.0,ladder_band,3378.0,18923.0,only 2 ACTIVE blocks (< 4) -- band rarely visi...
8,LKY/USDT,LKY-USDT,NonKYC,5m,SUSPECT,10+8,quantile,linear,near,-34.272,...,1.0000,200.000000,200,1.1869,15.0,204.0,ladder_band,204.0,26126.0,only 1 ACTIVE blocks (< 4) -- band rarely visi...
9,XVG/USDT,XVG-USDT,NonKYC,5m,SUSPECT,9+4,pct,back_loaded,near,-33.517,...,1.0000,200.000000,200,1.1096,1.0,25.0,ladder_band,25.0,9115.0,only 3 ACTIVE blocks (< 4) -- band rarely visi...



— HOLDOUT (the deciding out-of-sample evidence) —


,base,validation,holdout_edge_pct,holdout_trades,holdout_two_sided,holdout_active,edge_pct,clean_edge_pos_rate,harvest_best_pct_mo,max_fund,max_fund_quote,depth_2pct,gates
0,DASH/USDT,CONFIRMED,1.429,19,True,True,56.370,0.800,38.411,1000,1000.000000,2.0,
1,BELLS/USDT,CONFIRMED,17.425,102,True,True,43.578,0.800,32.373,700,700.000000,311.0,
2,BELLS/BTC,GATED,-1.721,54,True,True,47.682,0.714,24.713,200,0.003152,3.0,thin book ($55 in the min_band_5pct)
3,FET/USDT,GATED,2.669,46,True,True,35.719,0.800,29.488,300,300.000000,25090.0,rolling WF (process check) failed
4,BC2/USDT,SUSPECT,20.320,130,True,True,24.491,1.000,84.346,1000,1000.000000,676.0,only 2 ACTIVE blocks (< 4) -- band rarely visi...
5,EPIC/USDT,SUSPECT,-2.912,33,False,True,20.577,1.000,32.684,600,600.000000,23.0,clean: only 1 ACTIVE blocks (< 4) -- band rare...
6,PEP/USDC,SUSPECT,-3.998,37,False,True,5.110,1.000,32.398,200,199.422000,7.0,only 2 ACTIVE blocks (< 4) -- band rarely visi...
7,PEP/USDT,SUSPECT,-5.857,49,False,True,4.430,1.000,44.085,800,800.000000,138.0,only 2 ACTIVE blocks (< 4) -- band rarely visi...
8,LKY/USDT,SUSPECT,-15.896,29,True,True,2.853,1.000,29.094,200,200.000000,15.0,only 1 ACTIVE blocks (< 4) -- band rarely visi...
9,XVG/USDT,SUSPECT,-1.568,48,False,True,1.487,1.000,26.770,200,200.000000,1.0,only 3 ACTIVE blocks (< 4) -- band rarely visi...


  wrote artifacts/nonkyc/20260713-040416/files/NONKYC_recycle_v11_final_summary.csv
  wrote artifacts/nonkyc/20260713-040416/files/NONKYC_recycle_v11_walkforward_summary.csv
  wrote artifacts/nonkyc/20260713-040416/files/NONKYC_recycle_v11_live_yaml_summary.csv
  wrote artifacts/nonkyc/20260713-040416/files/NONKYC_recycle_v11_block_details.csv
  wrote artifacts/nonkyc/20260713-040416/files/NONKYC_recycle_v11_holdout_summary.csv
  wrote artifacts/nonkyc/20260713-040416/files/NONKYC_recycle_v11_deploy_config.json
  wrote artifacts/nonkyc/20260713-040416/files/NONKYC_recycle_v11_copy_paste_ladders.md
  wrote deployable YAML artifacts/nonkyc/20260713-040416/files/deploy_yamls/range_inventory_ladder_bells_usdt_auto_20260713.yml
  wrote deployable YAML artifacts/nonkyc/20260713-040416/files/deploy_yamls/range_inventory_ladder_dash_usdt_auto_20260713.yml
  wrote deployable YAML artifacts/nonkyc/20260713-040416/files/deploy_yamls/range_inventory_ladder_fet_usdt_auto_20260713.yml
  wrote deploy

In [10]:
# ── Cell 10: VERIFY THE BOOKS for the shortlist ───────────────────────
# v11.2: reports the full DEPTH PROFILE (+-2/5/10/25% + whole book) and the
# depth inside each market's OWN deployed ladder span. A fixed +-2% reading
# misjudges ladder-shaped books: NonKYC DASH/USDT quotes dust at the touch
# and parks real size ~2.6% out (+-2% = $2, +-5% = $29,820, book = $33,509).
short = final_df[((final_df.holdout_edge_pct > 1) &
                  (final_df.clean_edge_pos_rate >= 0.6)) |
                 (final_df.validation != 'SUSPECT')]
short_pairs = list(short.base.head(12))
print('verifying books for:', short_pairs)
lads = {pk: evals[pk]['deployed'] for pk in short_pairs if pk in evals}
book_df = lr.verify_books(uni, short_pairs, CFG, samples=5, pause=4.0, ladders=lads)
display(book_df)
book_df.to_csv(f'{PREFIX}_book_verification.csv', index=False)
print('READ THE PROFILE, not one band:')
print('  depth_2pct tiny but depth_5pct large -> dust quoted at the touch, real size')
print('     parked further out. NOT a thin market -- depth_ladder_band is the truth.')
print('  depth_ladder_band = liquidity inside YOUR rungs (what can actually fill them).')
print('  flaky=True -> book swinging >5x across samples; trust no single reading.')
print('  thin_med=True with steady samples -> genuinely thin: size down or skip.')


verifying books for: ['DASH/USDT', 'BELLS/USDT', 'BELLS/BTC', 'FET/USDT', 'BC2/USDT', 'NKYC/USDT', 'XLM/USDT', 'DYDX/USDT', 'XMR/USDT', 'XMR/USDC', 'BDX/BTC', 'BC2/BTC']
  book check DASH/USDT: {'market': 'DASH/USDT', 'samples_ok': 5, 'spread_pct': 0.8417, 'book_total_usd': 33535.0, 'n_bids': 67.0, 'n_asks': 78.0, 'depth_2pct': 2.0, 'depth_5pct': 30043.0, 'depth_10pct': 30057.0, 'depth_25pct': 30204.0, 'flaky': False, 'depth_ladder_band': 30066.0, 'thin_med': False, 'size_suggestion': 10000.0}
  book check BELLS/USDT: {'market': 'BELLS/USDT', 'samples_ok': 5, 'spread_pct': 1.0127, 'book_total_usd': 12626.0, 'n_bids': 72.0, 'n_asks': 100.0, 'depth_2pct': 311.0, 'depth_5pct': 548.0, 'depth_10pct': 1227.0, 'depth_25pct': 3062.0, 'flaky': False, 'depth_ladder_band': 1375.0, 'thin_med': False, 'size_suggestion': 688.0}
  book check BELLS/BTC: {'market': 'BELLS/BTC', 'samples_ok': 5, 'spread_pct': 1.0695, 'book_total_usd': 7076.0, 'n_bids': 43.0, 'n_asks': 58.0, 'depth_2pct': 3.0, 'depth_5pc

,market,samples_ok,spread_pct,book_total_usd,n_bids,n_asks,depth_2pct,depth_5pct,depth_10pct,depth_25pct,flaky,depth_ladder_band,thin_med,size_suggestion
0,DASH/USDT,5,0.8417,33535.0,67.0,78.0,2.0,30043.0,30057.0,30204.0,False,30066.0,False,10000.0
1,BELLS/USDT,5,1.0127,12626.0,72.0,100.0,311.0,548.0,1227.0,3062.0,False,1375.0,False,688.0
2,BELLS/BTC,5,1.0695,7076.0,43.0,58.0,3.0,55.0,63.0,80.0,False,12.0,True,6.0
3,FET/USDT,5,0.2524,48232.0,59.0,40.0,26260.0,48070.0,48095.0,48206.0,False,48206.0,False,10000.0
4,BC2/USDT,5,1.1885,8654.0,98.0,100.0,676.0,1744.0,1934.0,3304.0,False,1914.0,False,957.0
5,NKYC/USDT,5,0.1355,5016.0,100.0,100.0,2336.0,2441.0,2507.0,2552.0,False,2552.0,False,1276.0
6,XLM/USDT,5,0.8102,55823.0,73.0,66.0,38444.0,51332.0,51430.0,51554.0,False,51514.0,False,10000.0
7,DYDX/USDT,5,0.4677,4853975.0,41.0,44.0,9403.0,13104.0,15336.0,18067.0,False,18066.0,False,9033.0
8,XMR/USDT,5,0.2090,89893.0,100.0,100.0,58787.0,82030.0,86949.0,89893.0,False,86193.0,False,10000.0
9,XMR/USDC,5,1.3420,11780.0,45.0,41.0,9226.0,11704.0,11716.0,11762.0,False,11729.0,False,5865.0


READ THE PROFILE, not one band:
  depth_2pct tiny but depth_5pct large -> dust quoted at the touch, real size
     parked further out. NOT a thin market -- depth_ladder_band is the truth.
  depth_ladder_band = liquidity inside YOUR rungs (what can actually fill them).
  flaky=True -> book swinging >5x across samples; trust no single reading.
  thin_med=True with steady samples -> genuinely thin: size down or skip.


In [11]:
# ── Cell 11 (optional): calibrate the fill model against LIVE fills ───
# Point each pair at ONE OR MANY controller diagnostic JSONL files -- a single
# path, a list, and/or glob patterns. Rotated/overlapping logs are safe: fills
# are merged chronologically and deduplicated on (ts, side, price, amount).
#
# How much data is meaningful (rule of thumb: count error ~ 1/sqrt(N fills)):
#   < 15 fills / <10d  -> directional hint only, do NOT tune knobs
#   30+ fills, 15d+    -> first honest sim/live ratio (one full block)
#   60+ fills, 30d+    -> tune rc_volume_cap_frac / rc_fill_penetration_pct
#                         until sim_over_live_fill_ratio ~= 0.8-1.2
# If your controllers have been logging since deployment, you may already
# have months of data -- point globs at the whole log directory.
DIAG_FILES = {
    # 'XMR/USDT': '/path/to/xmr_diag.jsonl',                       # single file
    # 'XMR/USDT': ['/logs/xmr_*.jsonl', '/old_logs/xmr_2026*.jsonl'],  # globs+lists
}

DIAG_FILES = {
    'XMR/USDT': ['./diagnostics/nonkyc/range_inventory_ladder_xmr_usdt_diagnostic*.jsonl'],  # globs+lists
    'DASH/USDT': ['./diagnostics/nonkyc/range_inventory_ladder_dash_usdt_diagnostic*.jsonl'],  # globs+lists
    'ZANO/USDT': ['./diagnostics/nonkyc/range_inventory_ladder_zano_usdt_diagnostic*.jsonl'],  # globs+lists
    'SUN/USDT': ['./diagnostics/nonkyc/range_inventory_ladder_sun_usdt_diagnostic*.jsonl']  # globs+lists
}

for pk, paths in DIAG_FILES.items():
    print(f'=== {pk} ===')
    fills = lr.collect_jsonl_fills(paths)
    print(lr.fills_sufficiency(fills))
    if fills.empty:
        first = lr.first_existing(paths)          # glob-safe (v11.1.2 fix)
        if first is None:
            print(f'no files matched: {paths} -- check the path/glob')
        else:
            print(f'no fills extracted from {first} -- schema sniff:')
            display(lr.summarize_diagnostic_jsonl(first))
        continue
    print(fills.groupby('source_file').size().to_string())
    ladder = next((lr.controller_to_ladder(p, CFG) for p in yamls
                   if lr._pair_key(p['trading_pair']) == pk), None)
    h = hist_report.get(pk, hist_h.get(pk))
    if ladder is not None and h is not None:
        pdec = uni.get('pdec', {}).get(pk)
        book_half = lr.market_book_half_spread_pct(uni, pk, CFG)
        cfg_m, _ = lr.quote_scaled_cfg(uni, pk, CFG)
        cmp_ = lr.compare_live_vs_sim_v11(fills, h['bars'], ladder, cfg_m,
                                          pdec=pdec, book_half=book_half)
        for k, v in cmp_.items():
            print(f'  {k:28s} {v}')
    else:
        print('  (no matching controller YAML or history for the sim side)')

# refresh the health table with live-fill recency, now that fills are loaded
if DIAG_FILES and yaml_reports:
    fills_map = {pk: lr.collect_jsonl_fills(paths) for pk, paths in DIAG_FILES.items()}
    health = lr.live_strategy_health(yaml_df, yaml_reports, CFG, fills_by_pair=fills_map)
    lr.print_health_banners(health)
    display(health)
    health.to_csv(f'{PREFIX}_live_health.csv', index=False)


=== XMR/USDT ===
5 fills over 0.4d (~181/15d, +-45% count error) -> TOO THIN: directional hint only, do not tune knobs on this
source_file
./diagnostics/nonkyc/range_inventory_ladder_xmr_usdt_diagnostic_20260712-0816-20260712-081634.jsonl    5
  window_days                  0.4
  live_buy_fills               2
  live_sell_fills              3
  live_fills_per_day           12.1
  sim_buy_fills                3
  sim_sell_fills               6
  sim_fills_per_day            21.78
  sim_pnl_pct                  0.422
  sim_edge_pct                 -0.052
  vol_capped_fills             0
  fill_model                   v11
  fills_sufficiency            5 fills over 0.4d (~181/15d, +-45% count error) -> TOO THIN: directional hint only, do not tune knobs on this
  sim_over_live_fill_ratio     1.8
=== DASH/USDT ===
4 fills over 0.1d (~804/15d, +-50% count error) -> TOO THIN: directional hint only, do not tune knobs on this
source_file
./diagnostics/nonkyc/range_inventory_ladder_dash_usdt_dia

,controller,pair,status,recent_blocks,recent_days,recent_pnl_pct,recent_hold_pct,recent_edge_pct,recent_trades,recent_two_sided_blocks,last_block_pnl_pct,full_pnl_pct,full_edge_pct,days_since_live_fill
0,range_inventory_ladder_dash_V1,DASH/USDT,HEALTHY,3,45.0,6.25,-2.95,9.20,65,3,1.410,11.262,23.402,0.4
1,range_inventory_ladder_sun_V1,SUN/USDT,HEALTHY,3,45.0,7.47,-8.59,16.05,32,1,7.332,5.561,15.627,0.2
2,range_inventory_ladder_zano_V1,ZANO/USDT,HEALTHY,3,45.0,16.44,-1.57,18.00,38,3,6.872,40.167,49.423,0.5
3,range_inventory_ladder_xmr_V1,XMR/USDT,HEALTHY,3,45.0,18.92,-6.67,25.59,240,3,6.887,-9.067,34.457,0.4


## Reading the results (v11 evidence hierarchy)

Work down this list; each level is weaker evidence than the one above it.

1. **`holdout_edge_pct`** (Cell 9 holdout view / `*_holdout_summary.csv`) — the fitted,
   train-anchored ladder on 15 days it never saw. This is the only fully out-of-sample
   number for the geometry you deploy. Positive and two-sided ⇒ real signal; dormant
   (`holdout_active=False`) ⇒ no evidence either way, and the gate says so.
2. **Clean blocks** (`clean_edge_pos_rate`, `clean_worst_block_edge`) — the frozen report's
   blocks that do *not* overlap the fit window. The headline `edge_pct`/`edge_pos_rate`
   still include the fit window (kept for v10 comparability) — prefer the clean ones.
3. **Your live YAMLs** (Cell 6) — ground truth for the fill model itself. If the sim's
   trades/block is far from your JSONL fills, recalibrate in Cell 11 before trusting anything.
4. **Rolling WF** — validates the *fitting process*, not the deployed ladder. A wf_pass with
   a bad holdout means the process is fine but this particular fit is a bad draw: refit later.
5. **`harvest_best_pct_mo`** — model-free ceiling on what any grid can extract at cost.
   If it's near zero, no amount of ladder tuning will make the pair work.

`vol_capped_fills` > 0 tells you the book, not the price path, limited the sim — size DOWN
(`max_fund_value_quote`) rather than dismissing the pair. `fit_score_gap` large (≫ typical)
suggests the winner is a lucky draw among 240 candidates — prefer pairs where holdout,
clean blocks *and* WF agree.

**NonKYC note:** native 5m candles omit trade-less periods; the engine grid-snaps them (`candle_gap_fill` in the summary shows how much was missing). A high gap-fill fraction plus `vol_capped_fills` > 0 = a genuinely thin market: believe the capped numbers, not your eyes on the chart.

**Deployment discipline:** deploy only `CONFIRMED`, at the suggested `max_fund_value_quote`
or less, and treat the FIRST live 15-day block as the final gate — if it underperforms the
holdout badly, stop and recalibrate (Cell 11) instead of re-fitting harder.
